In [33]:
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [34]:
url = "https://jiji.ng/api_web/v1/listing?slug=houses-apartments-for-sale&page=1&webp=true"

response = requests.get(url)

data = response.json()

Task 1: Data Extraction


Requirements:
Write a Python script to send a GET request to the endpoint.


Use the requests library.

Extract the following key fields from each advert object:


title


region, region_name, region_parent_name


price_title (formatted price)


attrs fields


Property size


Bedrooms


Bathrooms


Furnishing status


is_boost (enterprise/diamond)


Convert the extracted data into a Pandas DataFrame.


Save the dataset locally as jiji_housing_raw.csv.





In [35]:
records = []

for item in data['adverts_list']['adverts']:

    attrs = item.get("attrs", [])

    bedrooms = None
    bathrooms = None
    size = None
    furnishing = None
    
    for attr in attrs:

        name = attr.get("name", "").lower()
        value = attr.get("value", "")

        if "bedroom" in name:
            bedrooms = value

        elif "bathroom" in name:
            bathrooms = value

        elif "property size" in name:
            size = value

        elif "furnishing" in name:
            furnishing = value
    records.append({
        "title": item.get("title"),
        "region": item.get("region"),
        "region_name": item.get("region_name"),
        "region_parent_name": item.get("region_parent_name"),
        "price": item.get("price_title"),
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "property_size": size,
        "furnishing": furnishing,
        "is_boost": item.get("is_boost")
    })

df = pd.DataFrame(records)

df.to_csv("data/jiji_housing_raw.csv", index=False)

df.head(10)
            

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,"9bdrm Block of Flats in Mercy Land Estate, Ali...","Lagos State, Alimosho",Alimosho,Lagos State,"₦ 53,000,000,000",9,6,600,Unfurnished,vip_gold
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,"₦ 180,000,000",6,7,300,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,"₦ 40,000,000",5,5,300,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,"₦ 120,000,000",11,11,1200,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,"₦ 25,000,000",8,4,5000,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),"₦ 450,000,000",4,4,2000,Semi-Furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,"₦ 40,000,000",6,2,250,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,"₦ 210,000,000",4,4,400,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,"₦ 320,000,000",5,5,400,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,"₦ 50,000,000",6,6,600,Unfurnished,diamond


Task 2: Data Cleaning

Objective:


Prepare the dataset for reliable analysis.


Questions to Guide Cleaning:

1. Missing Values


Which columns have missing values?


How will you handle missing property size, bedrooms, bathrooms, or furnishing?




In [36]:
#Missing Values

df.isnull().sum()

title                 0
region                0
region_name           0
region_parent_name    0
price                 0
bedrooms              0
bathrooms             0
property_size         0
furnishing            0
is_boost              0
dtype: int64

In [37]:
#Which columns have missing values?
#How will you handle missing property size, bedrooms, bathrooms, or furnishing?
df["bedrooms"] = df["bedrooms"].fillna(0)
df["bathrooms"] = df["bathrooms"].fillna(0)
df["property_size"] = df["property_size"].fillna("Unknown")
df["furnishing"] = df["furnishing"].fillna("Unknown")

df

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,"9bdrm Block of Flats in Mercy Land Estate, Ali...","Lagos State, Alimosho",Alimosho,Lagos State,"₦ 53,000,000,000",9,6,600,Unfurnished,vip_gold
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,"₦ 180,000,000",6,7,300,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,"₦ 40,000,000",5,5,300,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,"₦ 120,000,000",11,11,1200,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,"₦ 25,000,000",8,4,5000,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),"₦ 450,000,000",4,4,2000,Semi-Furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,"₦ 40,000,000",6,2,250,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,"₦ 210,000,000",4,4,400,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,"₦ 320,000,000",5,5,400,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,"₦ 50,000,000",6,6,600,Unfurnished,diamond


2. Price Cleaning


Convert the price from strings like “₦ 85,500,000” to float (85500000.00)


Are there outlier prices that should be removed?




In [38]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("₦", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price"] = pd.to_numeric(df["price"], errors="coerce")

df

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,"9bdrm Block of Flats in Mercy Land Estate, Ali...","Lagos State, Alimosho",Alimosho,Lagos State,53000000000,9,6,600,Unfurnished,vip_gold
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,180000000,6,7,300,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,40000000,5,5,300,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,120000000,11,11,1200,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,25000000,8,4,5000,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),450000000,4,4,2000,Semi-Furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,40000000,6,2,250,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,210000000,4,4,400,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,320000000,5,5,400,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,50000000,6,6,600,Unfurnished,diamond


In [39]:
df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors="coerce")
df["bathrooms"] = pd.to_numeric(df["bathrooms"], errors="coerce")
df

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
0,"9bdrm Block of Flats in Mercy Land Estate, Ali...","Lagos State, Alimosho",Alimosho,Lagos State,53000000000,9,6,600,Unfurnished,vip_gold
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,180000000,6,7,300,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,40000000,5,5,300,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,120000000,11,11,1200,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,25000000,8,4,5000,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),450000000,4,4,2000,Semi-Furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,40000000,6,2,250,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,210000000,4,4,400,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,320000000,5,5,400,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,50000000,6,6,600,Unfurnished,diamond


In [40]:
#exclude an outliers in dataframe

Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
Q1
Q3
IQR = Q3 -Q1
IQR

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = ( df['price'] < lower_bound) | ( df['price'] > upper_bound)

print(outliers.sum())

new_df = df[
    (df['price'] > lower_bound) & ( df['price'] < upper_bound)
] 

new_df

2


,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,180000000,6,7,300,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,40000000,5,5,300,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,120000000,11,11,1200,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,25000000,8,4,5000,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),450000000,4,4,2000,Semi-Furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,40000000,6,2,250,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,210000000,4,4,400,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,320000000,5,5,400,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,50000000,6,6,600,Unfurnished,diamond
10,5bdrm Townhouse/Terrace in Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,550000000,5,5,600,Semi-Furnished,enterprise


3. Categorical Cleaning


Standardize region, region_name, region_parent_name.


Convert furnishing type to consistent categories
 (e.g., Furnished / Semi-furnished / Unfurnished / Unknown)



In [41]:
new_df["furnishing"] = (
    new_df["furnishing"]
    .str.lower()
    .replace({
        "semi furnished": "Semi-Furnished",
        "furnished": "Furnished",
        "unfurnished": "Unfurnished"
    })
)

new_df["furnishing"] = new_df["furnishing"].fillna("Unknown")
new_df

,title,region,region_name,region_parent_name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,180000000,6,7,300,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,40000000,5,5,300,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,120000000,11,11,1200,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,25000000,8,4,5000,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),450000000,4,4,2000,semi-furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,40000000,6,2,250,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,210000000,4,4,400,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,320000000,5,5,400,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,50000000,6,6,600,Unfurnished,diamond
10,5bdrm Townhouse/Terrace in Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,550000000,5,5,600,semi-furnished,enterprise


In [42]:
new_df.rename(columns={
    "region": "Region",
    "region_name": "Region Name",
    "region_parent_name": "Region Parent Name"
    }, inplace=True)
new_df

,title,Region,Region Name,Region Parent Name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,180000000,6,7,300,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,40000000,5,5,300,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,120000000,11,11,1200,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,25000000,8,4,5000,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),450000000,4,4,2000,semi-furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,40000000,6,2,250,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,210000000,4,4,400,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,320000000,5,5,400,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,50000000,6,6,600,Unfurnished,diamond
10,5bdrm Townhouse/Terrace in Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,550000000,5,5,600,semi-furnished,enterprise


4. Attributes Extraction


Extract values for Bedrooms, Bathrooms, Property size from the attrs JSON list.



In [43]:
def extract_size(x):

    if pd.isna(x):
        return None

    nums = re.findall(r"\d+", str(x))

    if nums:
        return float(nums[0])

    return None

new_df["property_size"] = new_df["property_size"].apply(extract_size)
new_df

,title,Region,Region Name,Region Parent Name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,180000000,6,7,300.0,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,40000000,5,5,300.0,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,120000000,11,11,1200.0,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,25000000,8,4,5000.0,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),450000000,4,4,2000.0,semi-furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,40000000,6,2,250.0,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,210000000,4,4,400.0,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,320000000,5,5,400.0,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,50000000,6,6,600.0,Unfurnished,diamond
10,5bdrm Townhouse/Terrace in Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,550000000,5,5,600.0,semi-furnished,enterprise


5. Remove duplicates


Look for duplicate titles or identical rows.




In [44]:
new_df.drop_duplicates(inplace=True)
new_df

,title,Region,Region Name,Region Parent Name,price,bedrooms,bathrooms,property_size,furnishing,is_boost
1,"Furnished 6bdrm Duplex in Cooperative Estate, ...","Lagos State, Alimosho",Alimosho,Lagos State,180000000,6,7,300.0,Furnished,vip_gold
2,"5bdrm Block of Flats in Mercyland Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,40000000,5,5,300.0,Unfurnished,vip_gold
3,Furnished 11bdrm Block of Flats in Mercy Land ...,"Lagos State, Alimosho",Alimosho,Lagos State,120000000,11,11,1200.0,Furnished,vip_gold
4,"8bdrm Block of Flats in Ifesowapo Estate, Alim...","Lagos State, Alimosho",Alimosho,Lagos State,25000000,8,4,5000.0,Unfurnished,vip_gold
5,4bdrm Duplex in Gaduwa for sale,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),450000000,4,4,2000.0,semi-furnished,premium
6,6bdrm Bungalow in Ketu-Ikosi for sale,"Kosofe, Ketu-Ikosi",Ketu-Ikosi,Kosofe,40000000,6,2,250.0,Unfurnished,False
7,"4bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,210000000,4,4,400.0,Unfurnished,False
8,"5bdrm Duplex in Captains Court, Ado / Ajah for...","Ajah, Ado / Ajah",Ado / Ajah,Ajah,320000000,5,5,400.0,Unfurnished,False
9,6bdrm Bungalow in Ipaja / Ipaja for sale,"Ipaja, Ipaja / Ipaja",Ipaja / Ipaja,Ipaja,50000000,6,6,600.0,Unfurnished,diamond
10,5bdrm Townhouse/Terrace in Phase 1 for sale,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,550000000,5,5,600.0,semi-furnished,enterprise


6. Save cleaned dataset as:
 jiji_housing_cleaned.csv




In [45]:
new_df.to_csv("jiji_housing_cleaned.csv", index=False)

Task 3: Exploratory Data Analysis (EDA)


Objective:
Understand the structure, patterns, and distribution of the Nigerian housing marketplace on Jiji.


Your EDA Should Answer These Questions:


1. Price Analysis


What is the average house price in Nigeria?


Which state has the highest and lowest mean property price?




In [46]:
new_df["price"].mean()

np.float64(234611111.1111111)

In [47]:
state_price = (
    new_df.groupby("Region Name")["price"]
    .mean()
    .sort_values(ascending=False)
)
print("Highest Mean Price State:")
print(state_price.head())

print("\nLowest Mean Price State:")
print(state_price.tail())

Highest Mean Price State:
Region Name
Ikoyi S.W          570000000.0
Lekki Phase 1      550000000.0
Guzape District    510000000.0
Gaduwa             450000000.0
Ikate              400000000.0
Name: price, dtype: float64

Lowest Mean Price State:
Region Name
Alimosho         91250000.0
Ipaja / Ipaja    50000000.0
Aniocha South    40000000.0
Ketu-Ikosi       40000000.0
Life Camp        13000000.0
Name: price, dtype: float64


2. Property Size & Features


What is the distribution of property sizes?


Do houses with more bedrooms/bathrooms cost significantly more?




In [48]:
new_df["property_size"].value_counts()

property_size
300.0     3
400.0     2
600.0     2
200.0     2
1200.0    1
5000.0    1
2000.0    1
250.0     1
350.0     1
570.0     1
450.0     1
500.0     1
3699.0    1
Name: count, dtype: int64

In [49]:
#Listings Per State
new_df["Region Name"].value_counts()

Region Name
Alimosho           4
Ado / Ajah         2
Guzape District    2
Gaduwa             1
Ketu-Ikosi         1
Ipaja / Ipaja      1
Lekki Phase 1      1
Life Camp          1
Gwarinpa           1
Ologuneru          1
Aniocha South      1
Ikoyi S.W          1
Ikate              1
Name: count, dtype: int64

In [50]:
#Furnished vs Unfurnished
new_df.groupby("furnishing")["price"].mean()

furnishing
Furnished         1.043333e+08
Unfurnished       2.238462e+08
semi-furnished    5.000000e+08
Name: price, dtype: float64

3. Regional Trends


Which regions have the highest number of listings?


Which regions dominate premium property sales?



In [51]:
region_counts = new_df["Region Name"].value_counts()

print(region_counts.head(10))

Region Name
Alimosho           4
Ado / Ajah         2
Guzape District    2
Gaduwa             1
Ketu-Ikosi         1
Ipaja / Ipaja      1
Lekki Phase 1      1
Life Camp          1
Gwarinpa           1
Ologuneru          1
Name: count, dtype: int64


In [52]:
#Top Regions by Average Price
premium_regions = (
    new_df.groupby("Region Name")["price"]
    .mean()
    .sort_values(ascending=False)
)

premium_regions.head(10)

Region Name
Ikoyi S.W          570000000.0
Lekki Phase 1      550000000.0
Guzape District    510000000.0
Gaduwa             450000000.0
Ikate              400000000.0
Ado / Ajah         265000000.0
Ologuneru          100000000.0
Gwarinpa            95000000.0
Alimosho            91250000.0
Ipaja / Ipaja       50000000.0
Name: price, dtype: float64

4. Furnishing Analysis


Are furnished apartments more expensive on average?



In [53]:
furnishing_price = (
    new_df.groupby("furnishing")["price"]
    .mean()
    .sort_values(ascending=False)
)

print(furnishing_price)

furnishing
semi-furnished    5.000000e+08
Unfurnished       2.238462e+08
Furnished         1.043333e+08
Name: price, dtype: float64


5. Listing Type


Do boosted/enterprise listings have higher prices?




In [54]:
boost_price = (
    new_df.groupby("is_boost")["price"]
    .mean()
)

print(boost_price)

is_boost
False         141000000.0
diamond        50000000.0
enterprise    281500000.0
premium       490000000.0
vip           485000000.0
vip_gold       93000000.0
Name: price, dtype: float64


In [55]:
#Correlation Analysis (Bonus)
corr = new_df[
    [
        "price",
        "property_size",
        "bedrooms",
        "bathrooms"
    ]
].corr()

corr

,price,property_size,bedrooms,bathrooms
price,1.000000,-0.279796,-0.218481,-0.073723
property_size,-0.279796,1.000000,0.297145,0.091986
bedrooms,-0.218481,0.297145,1.000000,0.713636
bathrooms,-0.073723,0.091986,0.713636,1.000000


Task 5: Summary of Findings 


After running your analysis, your findings section can look like:

Lagos recorded the highest average property prices in the marketplace.
Abuja ranked second in premium housing listings.
States in northern Nigeria generally had lower average property prices.
Furnished properties were consistently priced higher than unfurnished properties.
Property size showed a positive relationship with house price.
Bedrooms and bathrooms moderately influenced pricing.
Enterprise/boosted listings had significantly higher average prices.
Premium pricing was concentrated in Lagos and Abuja luxury estates.

Task 6: Business Insights & Recommendations


Furnished luxury apartments generate the highest pricing power on Jiji.

    
Lagos contributes the largest revenue opportunity due to high listing volume and premium prices.

    
Abuja remains a strong secondary market for upscale housing.

    
Lower-performing states need targeted advertising and improved property presentation.

    
Sellers should include detailed property features, professional photos, and furnishing information.

    
Boosted listings attract greater visibility and are associated with higher-priced properties.

    
Real estate agencies should prioritize premium inventory in Lagos and Abuja while using promotional campaigns to grow demand in emerging regions.




In [56]:
new_df.to_csv("data/nigeria-housing-cleaned-dataset.csv", index=False)